In [ ]:
!pip install openai -q

In [ ]:
import openai
import nltk
import numpy as np
import pandas as pd
import string

from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
df = pd.read_csv('laptop_scenarios_story_text.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (19545, 6)
Columns: ['user_profile', 'budget_class', 'price_tier', 'story_text', 'upgrade_first', 'gain_class']


**Raw Text**

Raw text has Uppercase letters, punctuation, and stop words. These are noise and could reduce the quality of the TF-IDF features

In [ ]:
print("Raw Text")

for i in range(3):
    print(f"\nStory {i+1}:")
    print(df['story_text'].iloc[i])


Raw Text

Story 1:
My main problem with this Apple Ultrabook is simple: as a student, I keep finding the same limits over and over. The core specs are Intel Core i5 at 2.3GHz, 8GB of RAM, 128GB SSD, and Intel Integrated graphics. It has to keep up with study sessions, online lectures, writing, and too many tabs open at once. I keep running into the limits of the 128GB total storage, especially with 128GB SSD. I do like that it is an Apple Ultrabook running macOS, and the machine feels fairly light in a bag. For student use, I mainly want the machine to feel uncomplicated and dependable. Because the budget is low, I have to be selective and focus on the biggest pain point first.

Story 2:
For anyone curious, this Apple Ultrabook has been my machine for student use, and the experience is mixed. To be fair, not everything is bad. The 13.3-inch 2560x1600 display with IPS panel and Retina-like sharpness gives it some appeal, and the overall form factor is fairly light in a bag. On paper it 

**Preprocessing Steps**

each steps hsows its results


*   Tokenization splits the data
*   Lowercasing to convert words to lowercase to treat them the same
*   Stop word removal to remove common words that add noise and no useful meaning for clustering
*   Punctuation Removal to remove symbols taht add noise
*   Stemming to get words to their root form so all forms from the same word will be treated the same.








In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

example_text = df['story_text'].iloc[0]

print("Preprocessing")
print("-------------")

print("\nRaw Text:")
print(example_text)

#Tokenisation
tokens_step1 = nltk.word_tokenize(str(example_text))
print("\nAfter")
print(tokens_step1[:20], "...")

#Lowercasing
tokens_step2 = [w.lower() for w in tokens_step1]
print("\nAfter")
print(tokens_step2[:20], "...")

#Stopword removal
tokens_step3 = [w for w in tokens_step2 if w not in stop_words]
print("\nAfter")
print(tokens_step3[:20], "...")

#Punctuation removal
tokens_step4 = [w for w in tokens_step3 if w not in string.punctuation]
print("\nAfter")
print(tokens_step4[:20], "...")

#Stemming
tokens_step5 = [stemmer.stem(w) for w in tokens_step4]
print("\nAfter")
print(tokens_step5[:20], "...")

print("\nPreprocesses Text")
print(' '.join(tokens_step5))

Preprocessing
-------------

Raw Text:
My main problem with this Apple Ultrabook is simple: as a student, I keep finding the same limits over and over. The core specs are Intel Core i5 at 2.3GHz, 8GB of RAM, 128GB SSD, and Intel Integrated graphics. It has to keep up with study sessions, online lectures, writing, and too many tabs open at once. I keep running into the limits of the 128GB total storage, especially with 128GB SSD. I do like that it is an Apple Ultrabook running macOS, and the machine feels fairly light in a bag. For student use, I mainly want the machine to feel uncomplicated and dependable. Because the budget is low, I have to be selective and focus on the biggest pain point first.

After
['My', 'main', 'problem', 'with', 'this', 'Apple', 'Ultrabook', 'is', 'simple', ':', 'as', 'a', 'student', ',', 'I', 'keep', 'finding', 'the', 'same', 'limits'] ...

After
['my', 'main', 'problem', 'with', 'this', 'apple', 'ultrabook', 'is', 'simple', ':', 'as', 'a', 'student', ',', 'i

In [ ]:
def preprocess(text):
    tokens = nltk.word_tokenize(str(text))
    tokens = [w.lower() for w in tokens]
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [w for w in tokens if w not in string.punctuation]
    tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(tokens)

df['clean_story'] = df['story_text'].apply(preprocess)

**TF-IDF**

TF-IDF converts the cleaned words into numerical vectors to use them in analysis for models. if it is a high wieght, that nmeans it appears frequently in one story but rarley shows in the whole dataset, which makes it more useful for the model to place it in the correct cluster and differentiate between the clusters.

this shows the vector of a TF-IDF for one story, each number is foe one word in the story

In [ ]:
vectorizer = TfidfVectorizer(max_features=3500)
tfidf_matrix = vectorizer.fit_transform(df['clean_story'])
feature_names = vectorizer.get_feature_names_out()

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"\nSample feature names (first 20): {feature_names[:20]}")

story_vector = tfidf_matrix[0].toarray()[0]
top_indices = story_vector.argsort()[::-1][:10]
top_terms_story = [(feature_names[i], round(story_vector[i], 4)) for i in top_indices if story_vector[i] > 0]

print("\nTF-IDF vector for Story 1:")
print(f"{'Term':<20} {'TF-IDF Score'}")
for term, score in top_terms_story:
    print(f"{term:<20} {score}")

TF-IDF matrix shape: (19545, 603)

Sample feature names (first 20): ['00kg' '02kg' '03kg' '04kg' '05kg' '06kg' '07kg' '08kg' '09kg' '0ghz'
 '10' '1024gb' '10kg' '11' '11kg' '12' '128gb' '12gb' '12kg' '13']

TF-IDF vector for Story 1:
Term                 TF-IDF Score
appl                 0.3912
128gb                0.343
maco                 0.2565
ultrabook            0.2209
student              0.1992
keep                 0.1988
main                 0.1696
limit                0.1495
simpl                0.1443
mani                 0.1423


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(tfidf_matrix.toarray())

pca = PCA(n_components=0.65, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA components retained: {X_pca.shape[1]}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.4f}")

PCA components retained: 74
Total variance explained: 0.6510


In [ ]:
km_pp = KMeans(n_clusters=3, init='k-means++', random_state=42)
df['Cluster'] = km_pp.fit_predict(X_pca)

print("Cluster labels attached successfully.")
print("\nCluster distribution:")
print(df['Cluster'].value_counts().sort_index())

Cluster labels attached successfully.

Cluster distribution:
Cluster
0     1656
1    12440
2     5449
Name: count, dtype: int64


**Top Keywords per Cluster**

using the TF-IDF matrix, the average score of each word in all stories in each cluster. If the word had the highest score this means that it is the most important term in the cluster

In [ ]:
tfidf_array = tfidf_matrix.toarray()

def get_top_terms(cluster_id, top_n=10):
    indices = df[df['Cluster'] == cluster_id].index
    cluster_tfidf = tfidf_array[indices].mean(axis=0)
    top_indices = cluster_tfidf.argsort()[::-1][:top_n]
    return [feature_names[i] for i in top_indices]

print("Top Keywords per Cluster")
for c in sorted(df['Cluster'].unique()):
    print(f"\nCluster {c}: {get_top_terms(c)}")

Top Keywords per Cluster

Cluster 0: ['care', 'strict', 'one', 'mean', 'lot', 'hurt', 'decis', 'issu', 'make', 'intel']

Cluster 1: ['graphic', 'feel', 'intel', 'integr', 'machin', '4gb', 'use', '256gb', 'ram', '8gb']

Cluster 2: ['offic', 'still', 'work', 'want', 'feel', 'rather', 'meet', 'mostli', 'comfort', 'notic']


**Example Stories per Cluster**

example stories in each cluster shows kind of language used on that cluster here. To validate the we cpatured meaningul patterns in the data.

In [ ]:
print("Example Stories per Cluster")

for c in sorted(df['Cluster'].unique()):
    cluster_data = df[df['Cluster'] == c]
    print(f"CLUSTER {c}")
    examples = cluster_data['story_text'].iloc[:2]
    for i, story in enumerate(examples):
        print(f"\nExample {i+1}:")
        print(story[:300], "...")
        print("--------------")

Example Stories per Cluster
CLUSTER 0

Example 1:
My journal entry for this Apple Ultrabook is simple: it helps me get moving, but it also reminds me of its limits during a gamer day. The machine has to handle gaming, downloads, background apps, and the expectation that visuals stay smooth. The core specs are Intel Core i5 at 2.3GHz, 8GB of RAM, 12 ...
--------------

Example 2:
Today I spent another stretch of time with my Apple Ultrabook, trying to make it fit the way I work as a student. Most of my day is split between lectures, research tabs, note-taking, and assignments. The setup is built around an Intel Core i5 at 1.8GHz, paired with 8GB RAM, 128GB flash storage, and ...
--------------
CLUSTER 1

Example 1:
My main problem with this Apple Ultrabook is simple: as a student, I keep finding the same limits over and over. The core specs are Intel Core i5 at 2.3GHz, 8GB of RAM, 128GB SSD, and Intel Integrated graphics. It has to keep up with study sessions, online lectures, writing, 

# **GPT**

In [ ]:
from openai import OpenAI


**Explanation of what each clusters means**

We used here a RAG model to generate a cluater name, an upgrade meaning and the explanation for each cluster. We gave input for it: TF-IDF keywords for each cluster, and gave it a prompt to know the role of it and be able to generate an output based on what we ask or specify

In [ ]:
from google.colab import userdata
client = OpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=userdata.get("OPENROUTER_API_KEY"))
def generate_cluster_insight(cluster_id, top_terms):
    terms_str = ', '.join(top_terms)

    prompt = f"""
You are a data analyst working on a Smart Laptop Upgrade Advisor system.
You have clustered laptop user stories and extracted the following
top keywords from Cluster {cluster_id}:

Keywords: {terms_str}

Based only on these keywords, provide:
1. Cluster Name: A short 3-5 word name describing this user group
2. Likely Upgrade Meaning: What laptop upgrade these users most likely need
3. Explanation: 2-3 sentences explaining how the keywords support
   the cluster name and upgrade meaning

Format your response exactly like this:
Cluster Name: ...
Likely Upgrade Meaning: ...
Explanation: ...
"""

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

#Generate
results = []

for c in sorted(df['Cluster'].unique()):
    top_terms = get_top_terms(c, top_n=10)
    print(f"\nGenerating insight for Cluster {c}...")
    print(f"Top terms: {top_terms}")
    insight = generate_cluster_insight(c, top_terms)
    results.append({
        'Cluster ID': c,
        'Top Keywords': ', '.join(top_terms),
        'GPT Output': insight
    })
    print(insight)
    print("-------------------------------------------")


Generating insight for Cluster 0...
Top terms: ['care', 'strict', 'one', 'mean', 'lot', 'hurt', 'decis', 'issu', 'make', 'intel']
Cluster Name: Careful Decision Makers
Likely Upgrade Meaning: Upgrading to a more powerful Intel processor
Explanation: The keywords suggest that this user group is very careful and strict in their decision-making process. They are likely looking for a laptop upgrade that will address any issues they may have, such as a slow or outdated processor. Upgrading to a more powerful Intel processor would help them make faster and more efficient decisions.
-------------------------------------------

Generating insight for Cluster 1...
Top terms: ['graphic', 'feel', 'intel', 'integr', 'machin', '4gb', 'use', '256gb', 'ram', '8gb']
Cluster Name: Performance Enthusiasts
Likely Upgrade Meaning: Upgrading to a laptop with better graphics capabilities, higher RAM (8GB or more), and larger storage capacity (256GB or more).
Explanation: The keywords "graphic", "intel", "4

**Upgrade Evidence Sheet for each Cluster**

for each cluster we have:

*   cluster name from GPT
*   top TF-IDF keywords
*   Eamples from the dataset
*   the Common pattern using top keywords
*   upgrade meaning from GPT





In [ ]:
for c in sorted(df['Cluster'].unique()):
    cluster_data = df[df['Cluster'] == c]
    top_terms = get_top_terms(c, top_n=10)
    examples = cluster_data['story_text'].iloc[:2].tolist()
    gpt_output = results[c]['GPT Output']

    print("=" * 60)
    print(f"UPGRADE EVIDENCE SHEET – CLUSTER {c}")
    print("=" * 60)
    print(f"Top Keywords    : {', '.join(top_terms)}")
    print(f"\nExample Story 1 :")
    print(f"  {examples[0][:200]}...")
    print(f"\nExample Story 2 :")
    print(f"  {examples[1][:200]}...")
    print(f"\nCommon Patterns : Stories frequently use terms such as {', '.join(top_terms[:5])}.")
    print(f"\n{gpt_output}")
    print("=" * 60)
    print()

UPGRADE EVIDENCE SHEET – CLUSTER 0
Top Keywords    : care, strict, one, mean, lot, hurt, decis, issu, make, intel

Example Story 1 :
  My journal entry for this Apple Ultrabook is simple: it helps me get moving, but it also reminds me of its limits during a gamer day. The machine has to handle gaming, downloads, background apps, and ...

Example Story 2 :
  Today I spent another stretch of time with my Apple Ultrabook, trying to make it fit the way I work as a student. Most of my day is split between lectures, research tabs, note-taking, and assignments....

Common Patterns : Stories frequently use terms such as care, strict, one, mean, lot.

Cluster Name: Careful Decision Makers
Likely Upgrade Meaning: Upgrading to a more powerful Intel processor
Explanation: The keywords suggest that this user group is very careful and strict in their decision-making process. They are likely looking for a laptop upgrade that will address any issues they may have, such as a slow or outdated processor.

**Cluster 0**

smallest group, all users are low budget class, they use emotional phrases to show frustration, they need upgrades with financial constraints, they mostly need a RAM *upgrade*

**Cluster 1**

Largest group, they use technical words like laptop specification 'GB', they mainly gamers, designers and programmers. they also need a RAM upgrade and better graphics based on their workload

**Cluster 2**

~5k stories, most from officers. they use calm words and dont ask for upgrades immediatly.